In [1]:
import pandas as pd
from io import StringIO

In [2]:
## Since the same base model is used continually, I make my own empty df at the start
## ... to ensure the end result has its columns ordered correctly
params = {
    "OBSID":[],
    "phabs_nH":[],
    "highecut_cutoffE":[],
    "highecut_foldE":[],
    "powerlaw_PhoIndex":[],
    "powerlaw_norm":[],
    "bbody_kT":[],
    "bbody_norm":[],
    "gaussian_LineE":[],
    "gaussian_Sigma":[],
    "gaussian_norm":[]
}

df = pd.DataFrame(params)

In [3]:
def transcribe(obsid, xout, errout):
    
    ## define temporary arrays for restructuring values:
    frp = []                           ## list for holding frozen parameters
    params = {'OBSID':str(obsid)}      ## dict for storing params
                                       ## identifier column allows for stacking multiple OBSIDs (etc) 
                                       ## ... by applying the transcribe() function multiple times 
                                       ## ... and appending the outputs
    
    ## turn the Xspec parameters into a 2d array
    wt = xout.split('\n')
    wt = [line.split() for line in wt]
    
    ## check parameter array for frozen parameters
    for i in range(len(wt)-1):
        if wt[i][-1] == "frozen":
            frp += wt.pop(i)
            params[f'{frp[2]}_{frp[3]}'] = str(frp[-2])+'^f'
    
    ## turn the Xspec errors output into a 2d array
    errout = errout.split('\n')
    errout = [line.split() for line in errout]
    errout = [line[-1] for line in errout]
    errout = [line.replace('(', '').replace(')', '') for line in errout]
    errout = [line.split(',') for line in errout]
    
    ## define blank object into which typeset error bounds will be added
    errts = []
    
    for i in range(len(errout)):
        
        ## pull error upper and lower values and format them as exponents / subscripts
        errts += ['$_{' + errout[i][0] + '}^{+' + errout[i][1] + '}$']
        
        ## take parameter values from Xspec parameter array and append typeset errors
        wt[i][-3] = wt[i][-3]+errts[i]
    
    ## to param dict add entries for every parameter
    for line in wt:
        params[f'{line[2]}_{line[3]}'] = [line[-3]]

    ## param dict -> dataframe for nice formatting
    return pd.DataFrame(params)

In [4]:
## copy and paste of Xspec parameter output with no modifications yet
xp_out = '''   1    1   phabs      nH         10^22    7.51719      +/-  2.79193      
   2    2   highecut   cutoffE    keV      9.03644      +/-  0.893923     
   3    2   highecut   foldE      keV      29.0400      +/-  2.84688      
   4    3   powerlaw   PhoIndex            1.39564      +/-  8.84808E-02  
   5    3   powerlaw   norm                0.289371     +/-  6.59248E-02  
   6    4   bbody      kT         keV      0.290787     +/-  0.137229     
   7    4   bbody      norm                8.85753E-02  +/-  0.220390     
   8    5   gaussian   LineE      keV      6.41890      +/-  6.32368E-02  
   9    5   gaussian   Sigma      keV      0.100000     frozen
  10    5   gaussian   norm                1.99172E-03  +/-  5.08559E-04  '''

## copy and paste of Xspec error output with no modifications yet
xe_out = '''     1      4.70266      11.9306    (-2.82913,4.39885)
     2      7.93757      10.0965    (-1.09158,1.0674)
     3      27.2686      34.3171    (-1.77217,5.27633)
     4      1.27461      1.53468    (-0.121119,0.138948)
     5     0.200215     0.415684    (-0.0893198,0.12615)
     6    0.0979755     0.436074    (-0.193409,0.14469)
     7   0.00349305      0.71245    (-0.0844356,0.624521)
     8      6.33455       6.5046    (-0.0843003,0.0857515)
    10   0.00115552   0.00277727    (-0.0008342,0.000787545)'''

## obsid: 30001130002A

In [5]:
df = df.append(transcribe('30001130002A', xp_out, xe_out))
df

/var/folders/8b/k4k0b4yn01g09v5cc_lhcmpm0000gn/T/ipykernel_31440/1306745580.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(transcribe('30001130002A', xp_out, xe_out))


,OBSID,phabs_nH,highecut_cutoffE,highecut_foldE,powerlaw_PhoIndex,powerlaw_norm,bbody_kT,bbody_norm,gaussian_LineE,gaussian_Sigma,gaussian_norm
0,30001130002A,7.51719$_{-2.82913}^{+4.39885}$,9.03644$_{-1.09158}^{+1.0674}$,29.0400$_{-1.77217}^{+5.27633}$,1.39564$_{-0.121119}^{+0.138948}$,0.289371$_{-0.0893198}^{+0.12615}$,0.290787$_{-0.193409}^{+0.14469}$,8.85753E-02$_{-0.0844356}^{+0.624521}$,6.41890$_{-0.0843003}^{+0.0857515}$,0.100000^f,1.99172E-03$_{-0.0008342}^{+0.000787545}$
